In [3]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import os
import subprocess

df = pd.read_csv("diabetes.csv")

X = df.drop("Outcome", axis=1)
y = df["Outcome"]

print("Análise de Correlação com a variável alvo (Outcome):")
correlations = df.corr()["Outcome"].sort_values(ascending=False)
print(correlations)

plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Correlação das Features com o Outcome")
correlation_heatmap_path = "correlation_heatmap.png"
plt.savefig(correlation_heatmap_path)
plt.close()

print(f"Correlação heatmap salvo como {correlation_heatmap_path}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
y_pred = knn.predict(X_test_scaled)
initial_accuracy = accuracy_score(y_test, y_pred)
print(f"\nPrecisão inicial do modelo KNN (n_neighbors=5): {initial_accuracy:.4f}")

print("\nIniciando otimização de hiperparâmetros para o modelo KNN...")

param_grid = {
    'n_neighbors': list(range(1, 31)), 
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

knn_grid = KNeighborsClassifier()

grid_search = GridSearchCV(knn_grid, param_grid, cv=5, scoring='accuracy', n_jobs=-1)

grid_search.fit(X_train_scaled, y_train)

best_params = grid_search.best_params_
best_score = grid_search.best_score_

print(f"Melhores hiperparâmetros encontrados: {best_params}")
print(f"Melhor precisão (cross-validation) encontrada: {best_score:.4f}")

print("\nTreinando e avaliando o modelo KNN final com os melhores hiperparâmetros...")

final_knn_model = KNeighborsClassifier(**best_params)
final_knn_model.fit(X_train_scaled, y_train)

y_pred_final = final_knn_model.predict(X_test_scaled)

final_accuracy = accuracy_score(y_test, y_pred_final)
classification_rep = classification_report(y_test, y_pred_final)

print(f"Precisão final do modelo KNN no conjunto de teste: {final_accuracy:.4f}")
print("\nRelatório de Classificação:\n")
print(classification_rep)


Análise de Correlação com a variável alvo (Outcome):
Outcome                     1.000000
Glucose                     0.466581
BMI                         0.292695
Age                         0.238356
Pregnancies                 0.221898
DiabetesPedigreeFunction    0.173844
Insulin                     0.130548
SkinThickness               0.074752
BloodPressure               0.065068
Name: Outcome, dtype: float64
Correlação heatmap salvo como correlation_heatmap.png

Precisão inicial do modelo KNN (n_neighbors=5): 0.7013

Iniciando otimização de hiperparâmetros para o modelo KNN...
Melhores hiperparâmetros encontrados: {'metric': 'manhattan', 'n_neighbors': 15, 'weights': 'uniform'}
Melhor precisão (cross-validation) encontrada: 0.7688

Treinando e avaliando o modelo KNN final com os melhores hiperparâmetros...
Precisão final do modelo KNN no conjunto de teste: 0.7727

Relatório de Classificação:

              precision    recall  f1-score   support

           0       0.79      0.88  

In [7]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import os

df = pd.read_csv("diabetes.csv")

X = df.drop("Outcome", axis=1)
y = df["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
y_pred_initial = knn.predict(X_test_scaled)
initial_accuracy = accuracy_score(y_test, y_pred_initial)

param_grid = {
    'n_neighbors': list(range(1, 31)), # K de 1 ao 30
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

knn_grid = KNeighborsClassifier()
grid_search = GridSearchCV(knn_grid, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

best_params = grid_search.best_params_
best_score = grid_search.best_score_

final_knn_model = KNeighborsClassifier(**best_params)
final_knn_model.fit(X_train_scaled, y_train)
y_pred_final = final_knn_model.predict(X_test_scaled)
final_accuracy = accuracy_score(y_test, y_pred_final)
classification_rep_dict = classification_report(y_test, y_pred_final, output_dict=True)

plt.figure(figsize=(8, 6))
accuracies = [initial_accuracy, final_accuracy]
labels = ['Precisão Inicial (n_neighbors=5)', 'Precisão Final Otimizada']
sns.barplot(x=labels, y=accuracies, palette='viridis')
plt.ylim(0.6, 0.85) 
plt.title('Comparação de Precisão do Modelo KNN')
plt.ylabel('Precisão')
plt.savefig('accuracy_comparison.png')
plt.close()
print("Gráfico de comparação de precisão salvo como accuracy_comparison.png")

report_df = pd.DataFrame(classification_rep_dict).transpose()
report_df = report_df.drop(columns=['support'])
report_df = report_df.iloc[:-3] 

plt.figure(figsize=(10, 7))
sns.heatmap(report_df[['precision', 'recall', 'f1-score']], annot=True, cmap='Blues', fmt='.2f', linewidths=.5)
plt.title('Relatório de Classificação do Modelo KNN Otimizado')
plt.savefig('classification_report_heatmap.png')
plt.close()
print("Gráfico do relatório de classificação salvo como classification_report_heatmap.png")

results = pd.DataFrame(grid_search.cv_results_)

mean_test_scores = results.groupby('param_n_neighbors')['mean_test_score'].max()

plt.figure(figsize=(12, 7))
sns.lineplot(x=mean_test_scores.index, y=mean_test_scores.values, marker='o')
plt.title('Precisão de Validação Cruzada vs. Número de Vizinhos (n_neighbors)')
plt.xlabel('Número de Vizinhos (n_neighbors)')
plt.ylabel('Precisão Média de Validação Cruzada')
plt.xticks(mean_test_scores.index)
plt.grid(True)
plt.savefig('n_neighbors_accuracy.png')
plt.close()
print("Gráfico de precisão vs. n_neighbors salvo como n_neighbors_accuracy.png")

print("Geração de gráficos concluída.")




C:\Users\muckz\AppData\Local\Temp\ipykernel_12876\2999378608.py:49: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=labels, y=accuracies, palette='viridis')


Gráfico de comparação de precisão salvo como accuracy_comparison.png
Gráfico do relatório de classificação salvo como classification_report_heatmap.png
Gráfico de precisão vs. n_neighbors salvo como n_neighbors_accuracy.png
Geração de gráficos concluída.
